# SmartPlant AI — Verified Data Intake & 50-Class Training
**Research prototype · not a ready classifier.**

This notebook preserves the existing Android/Streamlit 29-class model. **Nothing is merged or deployed.**

This notebook intentionally checks *two branches*: the source-evidence review branch and the archived 50-class model development branch. The current production `main` was rolled back, so cloning only `main` will not provide the 50-class scripts.

Only proceed with training after dataset licensing, expert class labels, source plant/farm grouping and independent splits have been reviewed. Use photos that you own or are authorized to train on.

## 1 · Setup research and model source branches

This downloads **source code only**, not image datasets. Re-run in a fresh Colab runtime for reproducibility. Log both Git commits and the hardware environment.

In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content")
REVIEW = ROOT / "smartplant-data-review"
MODEL = ROOT / "smartplant-model-50"
REPO = "https://github.com/dapperdude404/tinyguard-ai.git"

def run(*args, cwd=None):
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)

def clone_branch(branch, target):
    if target.exists():
        if not (target / ".git").exists():
            raise RuntimeError(f"Folder exists but is not a clone: {target}")
        print("Using existing clone", target, "(restart runtime to refresh)")
    else:
        run("git", "clone", "--depth", "1", "--branch", branch, REPO, target)
    sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=target, text=True).strip()
    print(branch, "commit:", sha)
    return sha

review_commit = clone_branch("research/smartplant-50-dataset-intake", REVIEW)
model_commit = clone_branch("feature/smartplant-expanded-cnn-50", MODEL)

run(sys.executable, "-m", "pip", "install", "-q", "gdown", "pillow", "scikit-learn", "numpy")
import torch, torchvision
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA GPU available:", torch.cuda.is_available())

## 2 · Mount your own private image dataset

Expected: `MyDrive/SmartPlant50/data/expanded/`, with `train/c000..c049`, `val/c000..c049`, `test/c000..c049` and `groups.csv`. Every image needs a genuine source-group ID (plant or farm). No raw images are sent to GitHub.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/SmartPlant50/data/expanded")
print("Private dataset folder present:", DATA.is_dir())
print("No images are included with the GitHub repository.")

## 3 · View gaps before attempting training

The source register begins with **21 unapproved additions**. This diagnostic step prints what remains missing. A successful inventory structure is *not* proof of ground-truth labels or dataset independence.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", cwd=REVIEW)
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA, cwd=REVIEW)

## 4 · Hard source-evidence gate

STOP until a qualified reviewer has updated the source CSV with verified permissions, class mapping, original image counts and source groups. Do not set flags to "yes" without evidence. This cell is expected to fail for the initial 21 candidate entries.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
assert DATA.is_dir(), "Missing private dataset directory"
assert (DATA / "groups.csv").is_file(), "Missing plant/farm provenance file"
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA,
    "--require-complete", cwd=REVIEW)
print("Source evidence and structural inventory passed. Full image audit still required.")

## 5 · Generate and verify the original labels

Use the original SHA-256-pinned trained checkpoint. A *class plan* is just label metadata, not training. Generate it in the model repository first; do not overwrite a differing plan stored with your data.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
run(sys.executable, "-c",
    "from smartplant.model_download import ensure_cnn_checkpoint; "
    "print(ensure_cnn_checkpoint('models/smartplant_mobilenetv3.pt'))", cwd=MODEL)
run(sys.executable, "scripts/prepare_expanded_labels.py",
    "--checkpoint", "models/smartplant_mobilenetv3.pt",
    "--output", "data/expanded/class_plan.json", cwd=MODEL)
PLAN = MODEL / "data/expanded/class_plan.json"
assert PLAN.exists(), "Class plan generation failed"
if (DATA / "class_plan.json").is_file():
    assert PLAN.read_bytes() == (DATA / "class_plan.json").read_bytes(), (
        "Drive class plan differs from generated pinned plan: review label order"
    )
print("Reviewed 29→50 label plan generated. No model was trained.")

## 6 · Strict image audit

This performs image decoding, exact/near-duplicate screening and checks that source plant/farm IDs do not cross splits. The audit must PASS before training; test the same photographic source only in one partition. A hash screen cannot guarantee complete independence.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA,
    "--require-complete", cwd=REVIEW)
run(sys.executable, "scripts/audit_expanded_dataset.py", "--dataset", DATA,
    "--plan", PLAN, "--groups-csv", DATA / "groups.csv", cwd=MODEL)
print("Strict image audit passed. Independently validate labels and provenance.")

## 7 · Fine-tune the CNN (no cloud deployment)

Even if you execute this cell out of order, it re-runs all hard gates. Trains the existing MobileNetV3-Small on all 50 classes, selecting by validation macro-F1 and evaluating the test split once. GPU runtime recommended. Output remains in your private Drive.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
assert DATA.is_dir() and (DATA / "groups.csv").is_file(), "Dataset and provenance required"
assert (MODEL / "models/smartplant_mobilenetv3.pt").is_file(), "Verified source checkpoint missing"
assert (MODEL / "data/expanded/class_plan.json").is_file(), "Class plan missing"
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA,
    "--require-complete", cwd=REVIEW)
run(sys.executable, "scripts/audit_expanded_dataset.py", "--dataset", DATA,
    "--plan", MODEL / "data/expanded/class_plan.json", "--groups-csv", DATA / "groups.csv", cwd=MODEL)
OUTPUT = DATA.parent / "smartplant_50.pt"
run(sys.executable, "scripts/train_expanded_cnn.py", "--dataset", DATA,
    "--plan", MODEL / "data/expanded/class_plan.json",
    "--groups-csv", DATA / "groups.csv",
    "--base-checkpoint", MODEL / "models/smartplant_mobilenetv3.pt",
    "--output", OUTPUT, "--seed", "42", "--epochs", "25",
    "--batch-size", "32", "--patience", "6", "--lr", "0.0001", cwd=MODEL)
print("Research checkpoint in your private Drive:", OUTPUT)
print("Review its .metrics.json and .confusion.csv before considering any release.")

## 8 · Independent acceptance before Android/Streamlit updates

A trained model still needs external field photos, crop-mismatch checks (including the potato→strawberry failure case), a separate non-leaf rejection evaluation (fabric photo), subgroup error analysis, real-image PyTorch/ONNX parity, Android camera and airplane-mode testing.

**Never use synthetic 50-class smoke weights as a classifier.** There is no automatic GitHub merge, APK release or cloud deployment in this notebook.